# Colab T4 → VS Code tunnel (bootstrap)

Run this **in a Colab web tab** to expose the T4's Jupyter kernel so VS Code can attach to it. Then drive `finetune_qwen_colab.ipynb` from VS Code — cells execute on the T4.

**Steps:**
1. Open this notebook in Colab web, set `Runtime → Change runtime type → T4 GPU`.
2. Run both cells below. Copy the printed `URL/?token=...` line.
3. In VS Code: open `finetune_qwen_colab.ipynb` → click the kernel picker (top-right) → **Select Another Kernel → Existing Jupyter Server → Enter the URL of a running Jupyter server** → paste the line → pick the **Python 3** kernel.
4. Run the finetune notebook. `!git clone`, `import_qwen`, and training all run on Colab's T4.
5. **Keep this Colab tab open** — closing it (or ~90 min idle) kills the runtime and the tunnel. The URL changes every time you re-run.

**Note:** cells 8 & 9 of the finetune notebook (`google.colab` download/drive) won't work over an external kernel. To get `best.pt` back, open the same tunnel URL in a browser (it's a full Jupyter server), browse to `ARIA_LLM/checkpoints_chat_qwen/`, and download `best.pt`.

In [ ]:
# 1. Install cloudflared + a standalone Jupyter server.
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
!pip install -q notebook

In [ ]:
# 2. Start the Jupyter server (background) + a cloudflared quick tunnel,
#    then print the URL to paste into VS Code.
import subprocess, time, re, secrets

TOKEN = secrets.token_hex(16)

# Jupyter server, open to the tunnel, cross-origin allowed for VS Code.
jlog = open('/content/jupyter.log', 'w')
subprocess.Popen([
    'jupyter', 'notebook', '--ip=0.0.0.0', '--port=8888', '--no-browser', '--allow-root',
    f'--ServerApp.token={TOKEN}', f'--IdentityProvider.token={TOKEN}',
    '--ServerApp.allow_origin=*', '--ServerApp.disable_check_xsrf=True',
    '--ServerApp.allow_remote_access=True', '--ServerApp.root_dir=/content',
], stdout=jlog, stderr=subprocess.STDOUT)

# cloudflared quick tunnel -> prints a public https://*.trycloudflare.com URL.
clog_path = '/content/cloudflared.log'
clog = open(clog_path, 'w')
subprocess.Popen([
    'cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8888',
], stdout=clog, stderr=subprocess.STDOUT)

url = None
for _ in range(60):
    time.sleep(1)
    try:
        m = re.search(r'https://[-\w.]+\.trycloudflare\.com', open(clog_path).read())
    except FileNotFoundError:
        m = None
    if m:
        url = m.group(0)
        break

if not url:
    print('Tunnel did not come up — re-run this cell. Last log:')
    print(open(clog_path).read()[-1000:])
else:
    print('Tunnel is up. Paste THIS whole line into VS Code')
    print('(Jupyter: Existing Jupyter Server -> Enter the URL):\n')
    print(f'{url}/?token={TOKEN}')